In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from xgboost.sklearn import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.special import entr
from sklearn import preprocessing
import matplotlib.pyplot as plt
import math
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from IPython.display import display
import itertools
#from itables import init_notebook_mode
import random
#init_notebook_mode(all_interactive=True)

In [4]:
def generate_features(mouse_raw):
    x = mouse_raw['x'].copy()
    y = mouse_raw['y'].copy()
    horiz_spd = mouse_raw['x'].diff()/mouse_raw['timestamp'].diff()
    vert_spd = mouse_raw['y'].diff()/mouse_raw['timestamp'].diff()
    tang_spd = np.sqrt((horiz_spd**2)+(vert_spd**2))

    horiz_acc = horiz_spd.diff()/mouse_raw['timestamp'].diff()
    vert_acc = vert_spd.diff()/mouse_raw['timestamp'].diff()
    tang_acc = np.sqrt((horiz_acc**2)+(vert_acc**2))

    # Calculate the speed and acceleration of the mouse movement
    dx = np.diff(x)
    dy = np.diff(y)
    speed = np.sqrt(dx ** 2 + dy ** 2)
    acceleration = np.diff(speed)
    #acceleration = np.nan_to_num(acceleration)
    # Calculate the jerk of the mouse movement
    jerk = np.diff(acceleration)
    #jerk = np.mean(jerk)



    return { 'horiz_spd': horiz_spd, 'vert_spd': vert_spd, 'tang_spd':tang_spd,
             'horiz_acc': horiz_acc, 'vert_acc': vert_acc,
             'tang_acc': tang_acc, 'mouse_rawx_diff': mouse_raw['x'].diff(), 'mouse_rawx_timestamp': mouse_raw['timestamp'].diff(),
             'jerk': jerk }

def ranges(x):
    #print(x.max() - x.min())
    return x.max() - x.min()
def rmssd(x):
    return np.sqrt(np.mean(np.diff(x) ** 2))
def sdsd(x):
    return st.stdev(np.diff(x))
def nni_50(x):
    return  sum(np.abs(np.diff(x)) > 50)

def pnni_50(x):
    return 100 * nni_50(x) / len(x)

def nni_20(x):
    return sum(np.abs(np.diff(x)) > 20)

def pnni_20(x):
    return  100 * nni_20(x) / len(x)

def nni_5(x):
    return sum(np.abs(np.diff(x)) > 5)

def avg_hr(x):
    return  st.mean(100/x)

def avg_hr(x):
    return  st.mean(60000/x)
def std_hr(x):
    return  st.stdev(60000/x)
def min_hr(x):
    return  min(60000/x)
def max_hr(x):
    return  max(60000/x)

def energy(x):
    return sum(np.square(x))
def abs_sum_diff(x):
    return sum(np.abs(np.diff(x)))

def calculate_dwell_times(data, bin_size):
    # Create list of bin x and y coordinates
    bin_x = [i for i in range(0, 500, bin_size)]
    bin_y = [i for i in range(0, 500, bin_size)]

    # Function to calculate bin number of a point
    def calculate_bin(point, bin_list):
        return math.floor(point/bin_size)

    # Create new columns for bin numbers
    data['bin_x'] = data['x'].apply(calculate_bin, bin_list=bin_x)
    data['bin_y'] = data['y'].apply(calculate_bin, bin_list=bin_y)

    # Create new columns for bin coordinates
    data['bin_x_coord'] = data['bin_x'] * bin_size
    data['bin_y_coord'] = data['bin_y'] * bin_size

    # Create new column for dwell time
    data['dwell_time'] = 0

    # Loop through each row of dataframe
    for index, row in data.iterrows():
        # Skip first row
        if index == 0:
            continue

        # Get previous row
        prev_row = data.iloc[index-1]

        # Check if current row is in same bin as previous row
        if (row['bin_x'], row['bin_y']) == (prev_row['bin_x'], prev_row['bin_y']):
            # Calculate dwell time as difference between timestamps
            dwell_time = row['timestamp'] - prev_row['timestamp']
            data.at[index, 'dwell_time'] = dwell_time

    # Create new dataframe to store results
    results_df = pd.DataFrame(columns=['bin_x', 'bin_y', 'dwell_time'])

    # Loop through each unique bin_x, bin_y pair in original dataframe
    for bin_x, bin_y in data[['bin_x', 'bin_y']].drop_duplicates().values:
        # Filter dataframe to rows with current bin_x, bin_y pair
        bin_df = data[(data['bin_x'] == bin_x) & (data['bin_y'] == bin_y)]

        # If there's only one row, skip it
        if len(bin_df) == 1:
            continue

        # Sum the dwell time for all rows in current bin_x, bin_y pair
        total_dwell_time = bin_df['dwell_time'].sum()

        # Add current bin_x, bin_y pair and total dwell time to results dataframe
        results_df = results_df.append({'bin_x': bin_x, 'bin_y': bin_y, 'dwell_time': total_dwell_time}, ignore_index=True)

    return results_df

In [5]:
os.chdir('./datasets')
arr = os.listdir()
files_dict = {}


for i in arr:
    #print(i)
    print(i)
    if i == ".DS_Store":
        continue
    os.chdir(i)

    arr2 = os.listdir()
    print(len(arr2))
    #print(arr2)
    #print(len(arr2))
    #print(i)
    if len(arr2) != 0:#'./training' in arr2: len(arr2) != 0:
        #w = '/content/drive/MyDrive/boun dataset/'+ str(i) + '/training'
        print(arr2)
        w = './training'
        os.chdir(w)
        arr3 = os.listdir()
        files_dict[i] = arr3
        #print(w)
        #print(arr3)
        #print("----")
        os.chdir('../')

    os.chdir('../')
    print(os.listdir())


os.chdir('../')

1
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
10
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
11
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
12
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
13
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
14
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
15
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
16
1
['training']
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
17
1
['tr

In [6]:
user_dict = {}
user_dict_by_category = {}
all_category = ['office\n', 'file system\n', 'browsing\n', 'gaming\n', 'entertainment\n', 'development\n']
for i in list(files_dict.keys()):
    user_dict[i] = []
    user_dict_by_category[i] = {}
    for category in all_category:
        user_dict_by_category[i][category] = []

    #count = 1
    for j in list(files_dict[i]):

        #user_dict[i][count] = None
        w = './datasets/' + i + '/training/' + j
        file1 = open(w, 'r')
        Lines = file1.readlines()
        txt_array = []
        for k in range(0, len(Lines)):
            #print(Lines[k].split(","))
            txt_array.append(Lines[k].split(","))

        tmpp = pd.DataFrame(txt_array[1:], columns=txt_array[0])
        tmpp['timestamp'] = [int(i[0])*1000 + int(str(i[1])[:3].ljust(3, '0')) for i in tmpp['client_timestamp'].str.split('.')]
        uniquess = tmpp['window\n'].unique().tolist()
        #        print(uniquess)
        tmpp = tmpp[tmpp['window\n'] == 'browsing\n']
        user_dict[i].append(tmpp)
        for k in all_category:
            tmpp = tmpp[tmpp['window\n'] == k]
            #tmpp = tmpp[tmpp['state'] == "Move"]
            if tmpp.empty:
                user_dict_by_category[i][k].append(tmpp)

In [8]:
movement = {}

for i in list(user_dict.keys()):#[2:3]:#[:2]:
    movement[i] = []
    #for j in range(0, 5):
    for j in range(0, len(user_dict[i])):
        tt = user_dict[i][j]
        #print(tt)
        df = pd.DataFrame(tt, columns=tmpp.columns)
        #df = df[df["button"] == "MOVE"]
        #df['timestamp_mili'] = [int(i[0])*1000 + int(i[1]) for i in df['client_timestamp'].str.split('.')]
        movement[i].append(df)


In [ ]:
total_array = []

features_array = []
test = []
total_0 = 0
diff = 0
for i in list(movement.keys()):#[:1]:

    print(i)

    for k in range(0, len(movement[i])):#[3:4]:#[5:6]:
        print("Order", k)
        #for k in range(0, 1):#[3:4]:#[5:6]:

        df = pd.DataFrame(movement[i][k], columns=tmpp.columns)
        df.to_csv("testing.csv")
        tt = pd.read_csv("testing.csv")

        tt = tt[tt["state"] == "Move"]







        proper_a = []
        a = [d for _, d in tt.groupby(tt.index - np.arange(len(tt)))]
        #print(len(a))
        for u in a:
            first_timestamp = u['timestamp'].min()
            last_timestamp = u['timestamp'].max()
            time_passed = (last_timestamp - first_timestamp)/1000
            if time_passed > 8 and time_passed < 25:
                proper_a.append(u)
        #            if len(u) > 200:
        #print(len(u))
        #proper_a.append(u)


        a = proper_a
        #print(len(a))
        for kk in range(0, len(a)):
            print("Session", kk)
            features = {}
            features['user'] = i
            #for kk in range(0, 2):

            #if kk == 2:
            #    print(kk)
            xx = pd.DataFrame(a[kk], columns=tmpp.columns)
            xx = xx.drop_duplicates(subset='timestamp', keep='first')
            # xx.to_csv("testing.csv")
            # tt = pd.read_csv("testing.csv")





            xx['x'] = xx['x'].astype(int)
            xx['y'] = xx['y'].astype(int)
            xx['timestamp'] = xx['timestamp'].astype('int64')


            yyyx = xx.copy().reset_index()
            #print("----")
            #print(yyyx)
            #print("----")
            try:
                results = calculate_dwell_times(yyyx, 100)
            except:
                print(results)
                print("You entered here")
                continue
            #print("He")
            df = results
            # Convert dwell_time to numeric data type
            df['dwell_time'] = pd.to_numeric(df['dwell_time'])